In [ ]:
# 1. Desinstalar todas las librerías de LangChain y relacionadas.
!pip uninstall -y langchain langchain-community langchain-core langchain-google-genai langgraph langchain-text-splitters google-generativeai

# 2. Limpiar el caché de pip para asegurar que las nuevas instalaciones no usen paquetes corruptos.
!pip cache purge

In [ ]:
!pip uninstall -y langchain langchain-openai langchain-community faiss-cpu pypdf python-docx  streamlit python-dotenv

In [4]:
!pip cache purge

Files removed: 78


In [ ]:
!pip install -q \
    langchain --no-cache-dir \
    langchain-google-genai \
    google-generativeai \
    langchain_community \
    faiss-cpu \
    langchain-text-splitters \
    pymupdf \
    langgraph

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for documento in Path("/content/").glob("*.pdf"):
    try:
        loader = PyMuPDFLoader(str(documento))
        docs.extend(loader.load())
        print(f"Archivo cargado: {documento.name}")
    except Exception as e:
        print(f"Error cargando archivo: {documento.name}: {e}")

print(f"Total de documentos cargados: {len(docs)}")

In [3]:
print(len(docs))

3


**Dividir el texto**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
docs_splits = splitter.split_documents(docs)

**Crear Embeddings**

In [7]:
from google.colab import userdata

GEMINI_API_KEY=userdata.get("GEMINI_API_KEY")

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

modelo_embeddings = GoogleGenerativeAIEmbeddings(
    model = "models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

In [11]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs_splits, modelo_embeddings)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 4}
)

vectorstore.save_local("vectorstore")

In [13]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs_splits, modelo_embeddings)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 4}
)

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    google_api_key=GEMINI_API_KEY
)

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt_rag = ChatPromptTemplate(
    [
        ("system",
            """Eres el especialista en RR.HH. de la empresa Excelence Desarrollo de Software.
            Responde siempre utilizando los conocimientos del contexto que te fue pasado a ti.
            Si no hay informacion sobre la pregunta en los datos, responde solo 'No lo se'.
            """
        ),
        ("human", "Contexto: {context}\nPregunta del empleado: {input}")
    ]
)

# Helper function to format documents for stuffing into the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Re-implement create_stuff_documents_chain functionality using LCEL
document_chain = (
    {
        "context": lambda x: format_docs(x["context"]), # Takes list of docs, formats to string
        "input": RunnablePassthrough() # Passes the original 'input' through
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)

In [18]:
def consultar(pregunta):

    documentos = retriever.invoke(pregunta)

    contexto = "\n\n".join(
        [d.page_content for d in documentos]
    )

    mensajes = prompt_rag.format_messages(
        context=contexto,
        input=pregunta
    )

    respuesta = llm.invoke(mensajes)

    return respuesta.content

In [19]:
while True:

    p = input("Pregunta: ")

    if p=="salir":
        break

    print(consultar(p))

Pregunta: Como informar las vacaciones?
Las ausencias, incluyendo las solicitudes de vacaciones, deben gestionarse de la misma manera que si estuvieras trabajando en la oficina.
Pregunta: Como informar los gastos reembolsables?
Todos los gastos reembolsables deben ser presentados a través del sistema "Gestión de Gastos" antes del día 5 del mes siguiente. Es obligatorio adjuntar recibos o facturas legibles para todos los gastos, excepto la dieta diaria de viaje.
Pregunta: salir
